In [ ]:
!pip install transformers==4.40.0 IndicTransToolkit onnxruntime

In [ ]:
"""
Run this on Colab / Kaggle (GPU runtime).
Loads PRETRAINED AI4Bharat models — no training happens here.
Purpose: compare accuracy + latency on your Hindi/Marathi test clips
so you can decide which model/size to actually run live.

Before running:
  pip install -r requirements.txt
  Upload a few short .wav test clips (Hindi + Marathi, ideally including
  some code-switched sentences) to the Colab file browser.
"""

import time
import torch
import torchaudio
from transformers import AutoModel, AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
!pip install onnxruntime onnxruntime-gpu

In [ ]:
# ---------------------------------------------------------------------
# Step 1: Load pretrained ASR model (handles both Hindi 'hi' and Marathi 'mr')
# ---------------------------------------------------------------------
import os

ASR_MODEL_ID = "ai4bharat/indic-conformer-600m-multilingual"
print(f"Loading ASR model: {ASR_MODEL_ID}")

HF_TOKEN = os.environ.get("HF_TOKEN")  # set this in your shell / .env before running

asr_model = AutoModel.from_pretrained(
    ASR_MODEL_ID,
    trust_remote_code=True,
    token=HF_TOKEN,
).to(DEVICE)


def transcribe(audio_path: str, lang_code: str):
    """lang_code: 'hi' for Hindi, 'mr' for Marathi. Returns (transcript, latency_seconds)."""
    wav, sr = torchaudio.load(audio_path)
    wav = torch.mean(wav, dim=0, keepdim=True)  # mono
    if sr != 16000:
        wav = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(wav)

    start = time.time()
    transcript = asr_model(wav, lang_code, "ctc")
    latency = time.time() - start
    return transcript, latency


In [ ]:
# ---------------------------------------------------------------------
# Step 2: Load pretrained translation model (native language -> English)
# Swap to "ai4bharat/indictrans2-indic-en-dist-200M" for a lighter model
# if you want to test something closer to what you'd run locally.
# ---------------------------------------------------------------------
TRANS_MODEL_ID = "ai4bharat/indictrans2-indic-en-1B"
print(f"Loading translation model: {TRANS_MODEL_ID}")
trans_tokenizer = AutoTokenizer.from_pretrained(TRANS_MODEL_ID, trust_remote_code=True)
trans_model = AutoModelForSeq2SeqLM.from_pretrained(
    TRANS_MODEL_ID, trust_remote_code=True
).to(DEVICE)
ip = IndicProcessor(inference=True)


def translate_to_english(text: str, src_lang: str):
    """src_lang: 'hin_Deva' for Hindi, 'mar_Deva' for Marathi. Returns (english_text, latency_seconds)."""
    batch = ip.preprocess_batch([text], src_lang=src_lang, tgt_lang="eng_Latn")
    inputs = trans_tokenizer(batch, return_tensors="pt", padding=True).to(DEVICE)

    start = time.time()
    with torch.no_grad():
        generated = trans_model.generate(**inputs, max_length=256, num_beams=5, use_cache=False)
    latency = time.time() - start

    decoded = trans_tokenizer.batch_decode(generated, skip_special_tokens=True)
    output = ip.postprocess_batch(decoded, lang="eng_Latn")
    return output[0], latency

In [ ]:
# ---------------------------------------------------------------------
# Step 3: Run the benchmark on your test clips
# Fill this in with your actual file names. Set DATA_DIR to point at
# wherever your .wav clips live locally (defaults to ./data).
# ---------------------------------------------------------------------
import os

DATA_DIR = os.environ.get("DATA_DIR", "./data")

test_clips = [
    {"path": f"{DATA_DIR}/Marathi.wav", "lang": "mr", "trans_src": "mar_Deva"},
]

for clip in test_clips:
    transcript, asr_latency = transcribe(clip["path"], clip["lang"])
    english, trans_latency = translate_to_english(transcript, clip["trans_src"])

    print(f"\n--- {clip['path']} ---")
    print(f"Native transcript ({clip['lang']}): {transcript}")
    print(f"ASR latency: {asr_latency:.2f}s")
    print(f"English translation: {english}")
    print(f"Translation latency: {trans_latency:.2f}s")


In [ ]:
# ---------------------------------------------------------------------
# Step 3: Run the benchmark on your test clips
# Fill this in with your actual file names. Set DATA_DIR to point at
# wherever your .wav clips live locally (defaults to ./data).
# ---------------------------------------------------------------------
import os

DATA_DIR = os.environ.get("DATA_DIR", "./data")

test_clips = [
    {"path": f"{DATA_DIR}/Marathi.wav", "lang": "mr", "trans_src": "mar_Deva"},
    {"path": f"{DATA_DIR}/Ravi_Hindi.wav", "lang": "hi", "trans_src": "hin_Deva"},
    {"path": f"{DATA_DIR}/Ravi_Marathi.wav", "lang": "mr", "trans_src": "mar_Deva"},
    {"path": f"{DATA_DIR}/Tanvi_Marathi.wav", "lang": "mr", "trans_src": "mar_Deva"},
    {"path": f"{DATA_DIR}/Ishtiyaq_Hindi.wav", "lang": "mr", "trans_src": "mar_Deva"},
]

for clip in test_clips:
    transcript, asr_latency = transcribe(clip["path"], clip["lang"])
    english, trans_latency = translate_to_english(transcript, clip["trans_src"])

    print(f"\n--- {clip['path']} ---")
    print(f"Native transcript ({clip['lang']}): {transcript}")
    print(f"ASR latency: {asr_latency:.2f}s")
    print(f"English translation: {english}")
    print(f"Translation latency: {trans_latency:.2f}s")
